# 📊 VolNetX Comprehensive Evaluation
This notebook evaluates the VolNetX model using both **regression metrics** (RMSE, MAE, R²) and **classification metrics** (AUC-ROC, balanced accuracy, confusion matrix) for volatility direction prediction.
## Contents
1. Model & Data Loading
2. Inference on Validation Set
3. Regression Metrics (per horizon)
4. Direction Classification Metrics
5. Confusion Matrices & Classification Reports
6. Trading Strategy Backtest
7. Summary Table for CQF Report

In [1]:
# ==================================================
# 📦 Imports
# ==================================================
import os
import sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from tqdm.auto import tqdm
# Add repo root to path
sys.path.insert(0, os.path.abspath(".."))
# VolSense imports
from volsense_inference.model_loader import load_model
from volsense_core.evaluation.evaluation import ModelEvaluator
from volsense_execution.backtest import VolatilityDirectionBacktest, run_backtest_from_evaluator
# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)
plt.style.use("seaborn-v0_8-whitegrid")
print("✅ Imports complete")

c:\Users\rahul\OneDrive\Documents\GitHub\VolSense\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports complete


In [9]:
# ==================================================
# ⚙️ Configuration
# ==================================================
MODEL_VERSION = "volnetx"
CHECKPOINTS_DIR = "models"
DATASET_PATH = "data/processed/master_lstm_dataset_v2.csv"
VAL_START_DATE = "2024-01-01"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW = 65
HORIZONS = [1, 5, 10]

print(f"📌 Model: {MODEL_VERSION}")
print(f"📌 Device: {DEVICE}")  # Should print "cuda" on Colab
print(f"📌 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📌 GPU: {torch.cuda.get_device_name(0)}")

📌 Model: volnetx
📌 Device: cpu
📌 CUDA available: False


In [10]:
# ==================================================
# 🧠 Load VolNetX Model
# ==================================================
model, meta, scalers, ticker_to_id, features = load_model(
    MODEL_VERSION, 
    checkpoints_dir=CHECKPOINTS_DIR, 
    device=DEVICE
)
print(f"✅ Loaded model: {meta.get('arch', 'Unknown')}")
print(f"   Window: {meta.get('window', WINDOW)}")
print(f"   Horizons: {meta.get('horizons', HORIZONS)}")
print(f"   Features ({len(features)}): {features}")
print(f"   Tickers: {len(ticker_to_id)}")
print(f"   Scalers available: {scalers is not None}")

✅ Loaded model: VolNetX
   Window: 65
   Horizons: [1, 5, 10]
   Features (23): ['return', 'vol_10d', 'vol_20d', 'vol_60d', 'vol_3d', 'vol_vol', 'vol_ratio', 'vol_chg', 'abs_return', 'market_stress', 'vol_stress', 'rsi_14', 'macro_VIX', 'macro_CreditSpread', 'macro_Curve', 'macro_USD', 'event_earnings_heat', 'macro_Oil', 'macro_Rates', 'macro_BTC', 'return_sharpe_20d', 'skew_5d', 'skew_scaled_return']
   Tickers: 507
   Scalers available: True


In [6]:
# ==================================================
# 📂 Load Dataset
# ==================================================
df = pd.read_csv(DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
# Show date range
print(f"📊 Dataset shape: {df.shape}")
print(f"   Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"   Tickers: {df['ticker'].nunique()}")
# Filter to validation period
val_df = df[df["date"] >= VAL_START_DATE].copy()
print(f"\n📌 Validation set: {len(val_df):,} rows from {VAL_START_DATE}")
print(f"   Validation date range: {val_df['date'].min().date()} to {val_df['date'].max().date()}")

📊 Dataset shape: (2378068, 38)
   Date range: 2005-01-25 to 2025-10-23
   Tickers: 509

📌 Validation set: 229,646 rows from 2024-01-01
   Validation date range: 2024-01-01 to 2025-10-23


## 🔮 Model Inference
Running VolNetX inference on the validation set. For each ticker, we:
1. Extract the feature window (last 65 days)
2. Apply per-ticker scaling
3. Run forward pass to get predictions for all horizons

In [11]:
# ==================================================
# 🔮 Run Inference on Validation Set (OPTIMIZED)
# ==================================================

def run_volnetx_inference_fast(val_df, model, scalers, ticker_to_id, features, device, window=65, batch_size=64):
    """
    Fast VolNetX inference with batched processing.
    """
    model.eval()
    results = []
    
    val_dates = sorted(val_df["date"].unique())
    full_df = df.copy()
    
    for asof_date in tqdm(val_dates, desc="Running inference"):
        context_end = asof_date
        context_start = context_end - pd.Timedelta(days=window * 2)
        
        tickers_today = val_df[val_df["date"] == asof_date]["ticker"].unique()
        tickers_today = [t for t in tickers_today if t in ticker_to_id]
        
        if len(tickers_today) == 0:
            continue
        
        # Prepare batches
        batch_X = []
        batch_ticker_idx = []
        batch_tickers = []
        batch_today_vols = []
        
        for ticker in tickers_today:
            ticker_data = full_df[
                (full_df["ticker"] == ticker) & 
                (full_df["date"] <= context_end) &
                (full_df["date"] >= context_start)
            ].sort_values("date").tail(window)
            
            if len(ticker_data) < window:
                continue
            
            try:
                X = ticker_data[features].values.astype(np.float32)
            except KeyError:
                continue
            
            # Scale
            if scalers and ticker in scalers:
                scaler = scalers[ticker]
                X_tensor = torch.tensor(X, dtype=torch.float32)
                X_scaled = scaler.transform(X_tensor)
                if isinstance(X_scaled, torch.Tensor):
                    X_scaled = X_scaled.numpy()
            else:
                X_scaled = X
            
            batch_X.append(X_scaled)
            batch_ticker_idx.append(ticker_to_id[ticker])
            batch_tickers.append(ticker)
            
            asof_row = val_df[(val_df["date"] == asof_date) & (val_df["ticker"] == ticker)]
            today_vol = np.exp(asof_row["realized_vol_log"].values[0]) if not asof_row.empty else np.nan
            batch_today_vols.append(today_vol)
        
        if len(batch_X) == 0:
            continue
        
        # Stack into batch tensor [B, window, features]
        X_batch = torch.tensor(np.stack(batch_X), dtype=torch.float32).to(device)
        idx_batch = torch.tensor(batch_ticker_idx, dtype=torch.long).to(device)
        
        # Forward pass (batched!)
        with torch.no_grad():
            preds_batch = model(idx_batch, X_batch)  # [B, n_horizons]
        
        preds_np = preds_batch.cpu().numpy()
        
        # Collect results
        for b, ticker in enumerate(batch_tickers):
            today_vol = batch_today_vols[b]
            
            for i, h in enumerate(HORIZONS):
                future_date = asof_date + pd.Timedelta(days=h)
                future_row = val_df[(val_df["date"] >= future_date) & 
                                    (val_df["ticker"] == ticker)].head(1)
                realized_vol = np.exp(future_row["realized_vol_log"].values[0]) if not future_row.empty else np.nan
                forecast_vol = np.exp(preds_np[b, i])
                
                results.append({
                    "date": asof_date,
                    "ticker": ticker,
                    "horizon": h,
                    "forecast_vol": forecast_vol,
                    "realized_vol": realized_vol,
                    "today_vol": today_vol,
                })
    
    return pd.DataFrame(results)

# Run inference
print("🔮 Running VolNetX inference (batched)...")
eval_df = run_volnetx_inference_fast(
    val_df, model, scalers, ticker_to_id, features, DEVICE, WINDOW
)
print(f"✅ Inference complete: {len(eval_df):,} predictions")
print(f"   Horizons: {sorted(eval_df['horizon'].unique())}")
eval_df.head(10)

🔮 Running VolNetX inference (batched)...


Running inference:   0%|          | 1/662 [00:48<8:52:07, 48.30s/it]


KeyboardInterrupt: 

In [ ]:
# ==================================================
# 📊 Initialize ModelEvaluator
# ==================================================
# Drop rows with missing realized_vol (future dates not available)
eval_df_clean = eval_df.dropna(subset=["realized_vol", "forecast_vol"])
print(f"📊 Evaluation set: {len(eval_df_clean):,} valid predictions")
evaluator = ModelEvaluator(eval_df_clean, model_name="VolNetX")
print("✅ ModelEvaluator initialized")

## 📈 Regression Performance Metrics
Evaluating continuous volatility prediction using:
- **RMSE**: Root Mean Squared Error
- **MAE**: Mean Absolute Error  
- **MAPE**: Mean Absolute Percentage Error
- **R²**: Coefficient of Determination
- **Correlation**: Pearson correlation between forecast and realized

In [ ]:
# ==================================================
# 📈 Regression Metrics
# ==================================================
# Compute per-ticker, per-horizon metrics
metrics_df = evaluator.compute_metrics()
print(f"✅ Computed metrics for {len(metrics_df)} ticker-horizon combinations")
# Summarize by horizon
evaluator.summarize()

In [ ]:
# ==================================================
# 📊 Visualization: True vs Predicted
# ==================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, h in enumerate(HORIZONS):
    ax = axes[i]
    hdf = eval_df_clean[eval_df_clean["horizon"] == h]
    
    ax.scatter(hdf["realized_vol"], hdf["forecast_vol"], alpha=0.3, s=10)
    
    lims = [
        min(hdf["realized_vol"].min(), hdf["forecast_vol"].min()),
        max(hdf["realized_vol"].max(), hdf["forecast_vol"].max())
    ]
    ax.plot(lims, lims, "r--", label="Perfect Prediction")
    
    # Compute R² for title
    corr = np.corrcoef(hdf["realized_vol"], hdf["forecast_vol"])[0, 1]
    ax.set_title(f"{h}-Day Horizon (ρ = {corr:.3f})")
    ax.set_xlabel("Realized Volatility")
    ax.set_ylabel("Forecast Volatility")
    ax.legend()
plt.suptitle("VolNetX: Realized vs Forecast Volatility", fontsize=14)
plt.tight_layout()
plt.show()

## 🎯 Direction Classification Metrics
Per CQF requirements, we evaluate the model's ability to correctly classify **volatility direction** (increase vs decrease).
Direction is derived from regression predictions:
- `y_true = 1` if realized_vol > today_vol (volatility increased)
- `y_pred = 1` if forecast_vol > today_vol (model predicts increase)
Metrics computed:
- **AUC-ROC**: Area under ROC curve
- **Balanced Accuracy**: Accounts for class imbalance
- **Precision / Recall / F1**: Per-class metrics
- **Confusion Matrix**: Visual breakdown of predictions

In [ ]:
# ==================================================
# 🎯 Direction Classification Metrics
# ==================================================
# Compute direction metrics using the new ModelEvaluator methods
direction_metrics = evaluator.compute_direction_metrics(baseline_col="today_vol")
# Display summary
direction_summary = evaluator.summarize_direction()

In [ ]:
# ==================================================
# 📊 Confusion Matrices per Horizon
# ==================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, h in enumerate(HORIZONS):
    ax = axes[i]
    
    # Get confusion matrix
    hdf = eval_df_clean[eval_df_clean["horizon"] == h].copy()
    hdf["baseline_vol"] = hdf["today_vol"]
    hdf = hdf.dropna(subset=["baseline_vol", "forecast_vol", "realized_vol"])
    
    y_true = (hdf["realized_vol"] > hdf["baseline_vol"]).astype(int).values
    y_pred = (hdf["forecast_vol"] > hdf["baseline_vol"]).astype(int).values
    
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(y_true, y_pred)
    
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Vol Down", "Vol Up"],
        yticklabels=["Vol Down", "Vol Up"],
        ax=ax
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(f"{h}-Day Horizon")
plt.suptitle("VolNetX: Confusion Matrices for Volatility Direction", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ==================================================
# 📊 Classification Reports
# ==================================================
for h in HORIZONS:
    evaluator.classification_report_text(h, baseline_col="today_vol")

## 💰 Trading Strategy Backtest
Evaluating the practical utility of direction predictions via a simple volatility trading strategy:
- **Long volatility** when model predicts vol increase
- **Short volatility** when model predicts vol decrease
Performance metrics:
- Cumulative Return
- Sharpe Ratio
- Maximum Drawdown
- Win Rate

In [ ]:
# ==================================================
# 💰 Trading Strategy Backtest
# ==================================================
# Run backtest for 1-day horizon
bt = VolatilityDirectionBacktest(eval_df_clean)
result_1d = bt.run_direction_backtest(horizon=1)
# Display summary
bt.summary()

In [ ]:
# ==================================================
# 📊 Backtest Visualizations
# ==================================================
# Cumulative returns
bt.plot_cumulative_returns()
# Drawdown
bt.plot_drawdown()

In [ ]:
# ==================================================
# 📊 Backtest Comparison Across Horizons
# ==================================================
backtest_results = {}
for h in HORIZONS:
    bt_h = VolatilityDirectionBacktest(eval_df_clean)
    result = bt_h.run_direction_backtest(horizon=h)
    backtest_results[h] = result.metrics
# Create comparison table
bt_comparison = pd.DataFrame(backtest_results).T
bt_comparison.index.name = "Horizon"
bt_comparison = bt_comparison.reset_index()
print("\n📊 Backtest Comparison Across Horizons")
print("=" * 70)
display(bt_comparison.style.format({
    "total_return": "{:.2%}",
    "annualized_return": "{:.2%}",
    "annualized_volatility": "{:.2%}",
    "sharpe_ratio": "{:.2f}",
    "sortino_ratio": "{:.2f}",
    "max_drawdown": "{:.2%}",
    "win_rate": "{:.2%}",
    "profit_factor": "{:.2f}",
}))

## 📋 Summary Table for CQF Report
The following tables provide formatted output suitable for inclusion in the LaTeX report.

In [ ]:
# ==================================================
# 📋 Generate LaTeX Tables for Report
# ==================================================
print("=" * 70)
print("📊 REGRESSION METRICS TABLE (Copy to LaTeX)")
print("=" * 70)
# Aggregate by horizon
summary = evaluator.metrics_df.groupby("horizon").agg({
    "RMSE": "mean",
    "MAE": "mean",
    "MAPE": "mean",
    "R2": "mean",
    "Corr": "mean",
}).round(4)
print(summary.to_latex())
print("\n" + "=" * 70)
print("📊 DIRECTION CLASSIFICATION TABLE (Copy to LaTeX)")
print("=" * 70)
direction_table = pd.DataFrame([
    {
        "Horizon": f"{h}d",
        "AUC-ROC": m["auc_roc"],
        "Accuracy": m["accuracy"],
        "Balanced Acc.": m["balanced_accuracy"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "F1": m["f1"],
    }
    for h, m in evaluator.direction_metrics.items()
])
print(direction_table.to_latex(index=False, float_format="%.4f"))
print("\n" + "=" * 70)
print("📊 BACKTEST RESULTS TABLE (Copy to LaTeX)")
print("=" * 70)
backtest_table = pd.DataFrame([
    {
        "Horizon": f"{h}d",
        "Total Return": f"{m['total_return']:.2%}",
        "Ann. Sharpe": f"{m['sharpe_ratio']:.2f}",
        "Max DD": f"{m['max_drawdown']:.2%}",
        "Win Rate": f"{m['win_rate']:.2%}",
    }
    for h, m in backtest_results.items()
])
print(backtest_table.to_latex(index=False))

In [ ]:
# ==================================================
# ✅ Final Summary
# ==================================================
print("\n" + "=" * 70)
print("🎯 VOLNETX EVALUATION SUMMARY")
print("=" * 70)
print("\n📈 REGRESSION PERFORMANCE")
print("-" * 40)
for h in HORIZONS:
    h_metrics = evaluator.metrics_df[evaluator.metrics_df["horizon"] == h]
    print(f"  {h}d: RMSE={h_metrics['RMSE'].mean():.4f}, R²={h_metrics['R2'].mean():.4f}")
print("\n🎯 DIRECTION CLASSIFICATION")
print("-" * 40)
for h, m in evaluator.direction_metrics.items():
    print(f"  {h}d: AUC={m['auc_roc']:.4f}, Bal.Acc={m['balanced_accuracy']:.4f}")
print("\n💰 TRADING BACKTEST")
print("-" * 40)
for h, m in backtest_results.items():
    print(f"  {h}d: Sharpe={m['sharpe_ratio']:.2f}, Return={m['total_return']:.2%}")
print("\n" + "=" * 70)
print("✅ Evaluation complete! Results ready for CQF report.")
print("=" * 70)

In [ ]:
# ==================================================
# 💾 Save Results (Optional)
# ==================================================
# Save evaluation DataFrame
eval_df_clean.to_csv("data/processed/volnetx_eval_results.csv", index=False)
print("💾 Saved evaluation results to data/processed/volnetx_eval_results.csv")
# Save direction metrics
direction_summary.to_csv("data/processed/volnetx_direction_metrics.csv", index=False)
print("💾 Saved direction metrics to data/processed/volnetx_direction_metrics.csv")